# Lesson 3 · Phase 1 — Prompt Foundations: Zero-shot, Role & Few-shot

**Mastering Agentic AI Certification · Pre-read**

> The model's weights are frozen after training (Lessons 1–2). At **inference time** the *only* thing you control is the **prompt** — the text you feed in. **Prompt engineering** is the craft of shaping that text to steer the model's output. This phase covers the three foundational patterns.

## 1. The complete picture — anatomy of a prompt

```
  ┌──────────────────────────────────────────────┐
  │  ROLE / SYSTEM   "You are a senior tax advisor"│  ← who the model should be
  │  INSTRUCTION     "Classify the sentiment"      │  ← what to do
  │  EXAMPLES        input→output demonstrations   │  ← few-shot (optional)
  │  INPUT           "The food was cold."          │  ← the actual task
  └──────────────────────────────────────────────┘
                       │  (frozen weights — no training!)
                       ▼
                   MODEL  ──▶  output
```

**Key idea:** prompting changes *behaviour without changing weights*. It is **in-context conditioning**, not learning. Everything in this lesson is built from rearranging the blocks above.

## 2. Zero-shot prompting

**Just ask.** No examples — you rely entirely on what the model learned in pre-training/fine-tuning.

```
Classify the sentiment as positive, negative, or neutral.

Text: "The food was cold and the service was slow."
Sentiment:
```

- ✅ Simplest, cheapest (fewest tokens), fast to write.
- ⚠️ Can be unreliable for unusual formats, niche tasks, or strict output shapes.
- **Use when:** the task is common and the instruction is clear.

## 3. Role (persona / system) prompting

Assign the model an **identity** so it adopts the right tone, expertise, and assumptions. Usually placed in the **system message**.

```
SYSTEM: You are a meticulous senior security engineer. You are concise,
        cite risks explicitly, and never guess — you say when unsure.
USER:   Is it safe to run a public, token-less Jupyter server?
```

- Shifts vocabulary, depth, format, and "default" behaviour.
- Pairs with every other technique (you can role-prompt + few-shot + CoT together).
- **Use when:** you want a consistent voice, domain expertise, or guardrails.

## 4. Few-shot prompting

Show a handful of **input → output examples** ("shots") *inside the prompt*. The model infers the pattern and continues it — **in-context learning**.

```
Text: "Loved it!"            Sentiment: positive
Text: "Worst purchase ever." Sentiment: negative
Text: "It arrived on time."  Sentiment: neutral
Text: "The food was cold."   Sentiment:        ← model completes this
```

> **Link to earlier lessons:** the examples look like the *(input, label)* pairs from Lesson 1 — but **no weights change** (Lesson 2's gradients are *not* run). The model is merely *conditioned* by the context. That's why it's called few-shot *learning* in quotes — it's pattern-matching at inference time.

- ✅ Big reliability boost for format-sensitive or unusual tasks.
- ⚠️ Costs tokens; examples must be representative and well-formatted.

In [ ]:
# Prompt BUILDERS for the three foundational patterns.
# (No real LLM here — we assemble the exact text you'd send to the API.)

def zero_shot(task, text):
    return f"{task}\n\nText: \"{text}\"\nSentiment:"

def role_prompt(persona, task, text):
    return f"SYSTEM: {persona}\n\nUSER: {task}\nText: \"{text}\"\nSentiment:"

def few_shot(task, examples, text):
    shots = "\n".join(f'Text: "{t}"  Sentiment: {lab}' for t, lab in examples)
    return f"{task}\n\n{shots}\nText: \"{text}\"  Sentiment:"

TASK = "Classify the sentiment as positive, negative, or neutral."
EX = [("Loved it!", "positive"),
      ("Worst purchase ever.", "negative"),
      ("It arrived on time.", "neutral")]
INPUT = "The food was cold and the service was slow."

print("================ ZERO-SHOT ================")
print(zero_shot(TASK, INPUT))
print("\n================ ROLE ================")
print(role_prompt("You are a precise sentiment classifier. Reply with one word.", TASK, INPUT))
print("\n================ FEW-SHOT ================")
print(few_shot(TASK, EX, INPUT))

In [ ]:
# A tiny MOCK model to FEEL the effect of few-shot on output format.
# It is a keyword classifier -- NOT a real LLM -- just to make the point concrete.
def mock_model(prompt):
    text = prompt.lower().split('sentiment:')[0]   # crude: look at the input text
    last = prompt.rsplit('text:', 1)[-1].lower()
    neg = any(w in last for w in ["cold", "slow", "worst", "bad", "broke"])
    pos = any(w in last for w in ["loved", "great", "good", "excellent"])
    label = "negative" if neg else "positive" if pos else "neutral"
    # If the prompt showed few-shot examples, the model mimics their tidy one-word style.
    if "loved it!" in prompt.lower():
        return label
    return f"The sentiment seems {label}, though it depends on interpretation."  # chattier zero-shot

print("zero-shot output :", mock_model(zero_shot(TASK, INPUT)))
print("few-shot  output :", mock_model(few_shot(TASK, EX, INPUT)), " <- clean, matches the example format")

## 5. Which foundational pattern when?

| Pattern | Tokens | Reliability | Best for |
|---|---|---|---|
| **Zero-shot** | lowest | medium | common, clearly-stated tasks |
| **Role** | low | medium–high | consistent tone/expertise/guardrails |
| **Few-shot** | higher | high | strict formats, niche tasks, edge cases |

These **stack**: a strong prompt is often *role + few-shot + a clear instruction*. The reasoning techniques in **Phase 2** add to this base.

## 6. How this contributes to agentic AI

- An agent's every step is a **prompt** to the model — these patterns are the agent's basic control surface.
- **Role prompts** define an agent's persona and guardrails (e.g. "you are a careful coding agent that runs tests before claiming success").
- **Few-shot examples** teach an agent the exact **tool-call / output format** it must emit, without fine-tuning.
- They are the foundation the **reasoning** (Phase 2) and **agentic** (Phase 3) patterns build on.

## 7. Key takeaways

1. Prompting steers a **frozen** model — it is **in-context conditioning, not training** (no gradients).
2. **Zero-shot** = ask directly; cheapest, good for common tasks.
3. **Role prompting** sets identity/tone/guardrails via the system message.
4. **Few-shot** supplies in-prompt examples → big reliability gain for format-sensitive tasks.
5. The patterns **compose**, and they are the building blocks for reasoning and agentic prompting.

---
*Next (Phase 2):* getting the model to **reason** — Chain of Thought, Self-Consistency, Tree of Thoughts.

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0]); print("Platform:", platform.platform())
print("Lesson 3 · Phase 1 (Zero-shot, Role, Few-shot) notebook is running ✓")